# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [63]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [64]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

temp_date = pd.to_datetime(dim_tiempo['Release Date'], format='%m/%d/%Y')

dim_tiempo['Year'] = temp_date.dt.year
dim_tiempo['Month'] = temp_date.dt.month
dim_tiempo['Day'] = temp_date.dt.day

display(dim_tiempo.head(3))

,Release Date,id_tiempo,Year,Month,Day
0,4/26/2024,1,2024,4,26
1,5/4/2024,2,2024,5,4
2,3/19/2024,3,2024,3,19


In [65]:
# 2. Crear dim_pista
# dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True).copy()
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].copy()
dim_pista['id_track'] = dim_pista.index + 1
display(dim_pista.head(10))

,Track,Artist,Explicit Track,id_track
0,MILLION DOLLAR BABY,Tommy Richman,0,1
1,Not Like Us,Kendrick Lamar,1,2
2,i like the way you kiss me,Artemas,0,3
3,Flowers,Miley Cyrus,0,4
4,Houdini,Eminem,1,5
5,Lovin On Me,Jack Harlow,1,6
6,Beautiful Things,Benson Boone,0,7
7,Gata Only,FloyyMenor,1,8
8,Danza Kuduro - Cover,MUSIC LAB JPN,1,9
9,BAND4BAND (feat. Lil Baby),Central Cee,1,10


In [66]:
# 3. Crear fact_streams
df_temp = spotify_raw.merge(dim_tiempo, on='Release Date', how='left')
df_temp = df_temp.merge(dim_pista, on='Track', how='left')

fact_streams = df_temp[[
    'id_track',
    'id_tiempo',
    'Spotify Streams',
    'YouTube Views',
    'Track Score'
    ]]
display(fact_streams.head(3))

,id_track,id_tiempo,Spotify Streams,YouTube Views,Track Score
0,1,1,"390,470,936","84,274,754",725.4
1,2,2,"323,703,884","116,347,040",545.9
2,3,3,"601,309,283","122,599,116",538.4


In [67]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_tiempo, on='id_tiempo').merge(dim_pista, on='id_track')
cubo_olap['Release Date'] = pd.to_datetime(cubo_olap['Release Date'], format='%m/%d/%Y')
display(cubo_olap.head(3))

,id_track,id_tiempo,Spotify Streams,YouTube Views,Track Score,Release Date,Year,Month,Day,Track,Artist,Explicit Track
0,1,1,"390,470,936","84,274,754",725.4,2024-04-26,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0
1,2,2,"323,703,884","116,347,040",545.9,2024-05-04,2024,5,4,Not Like Us,Kendrick Lamar,1
2,3,3,"601,309,283","122,599,116",538.4,2024-03-19,2024,3,19,i like the way you kiss me,Artemas,0


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [76]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum()
print(respuesta_1.head(5))

Artist
"XY"                                                           0
$OHO BANI                                             54,065,563
$uicideboy$    44,027,794675,082,623189,588,27073,979,851204,...
&ME                                                   34,601,626
(G)I-DLE             289,304,213286,016,224242,708,36858,909,647
Name: Spotify Streams, dtype: object


In [77]:
# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
print(respuesta_2.head(10))

    id_track  id_tiempo Spotify Streams YouTube Views  Track Score  \
0          1          1     390,470,936    84,274,754        725.4   
1          2          2     323,703,884   116,347,040        545.9   
2          3          3     601,309,283   122,599,116        538.4   
7          5          5     107,034,922    77,373,957        423.3   
8         47          5     107,034,922    77,373,957        423.3   
9       1667          5     107,034,922    77,373,957        423.3   
12         7          7     900,158,751   308,723,145        407.2   
13         8          8     675,079,153   228,382,568        375.8   
14       224          8     675,079,153   228,382,568        375.8   
15         9          9   1,653,018,119           NaN        355.7   

   Release Date  Year  Month  Day                       Track          Artist  \
0    2024-04-26  2024      4   26         MILLION DOLLAR BABY   Tommy Richman   
1    2024-05-04  2024      5    4                 Not Like Us  Kend

In [78]:
# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down
respuesta_3 = cubo_olap.groupby(['Year', 'Month'])['Spotify Streams'].sum()
print(respuesta_3.head(10))


Year  Month
1987  7                                  1,879,386,440
1991  9                     2,021,910,3642,021,910,364
1994  1                                  1,810,649,765
1998  4                                    178,339,925
1999  6                                  1,405,353,677
2000  1                                  1,798,020,679
      10                    2,045,512,4212,045,512,421
2001  9                                    399,145,178
2002  1        2,146,682,3422,146,682,3422,146,682,342
      5                                  1,958,809,069
Name: Spotify Streams, dtype: object


In [79]:
# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['Year'] == 2024)]
print(respuesta_4.head(10))


    id_track  id_tiempo Spotify Streams YouTube Views  Track Score  \
1          2          2     323,703,884   116,347,040        545.9   
7          5          5     107,034,922    77,373,957        423.3   
13         8          8     675,079,153   228,382,568        375.8   
15         9          9   1,653,018,119           NaN        355.7   
16        23          9   1,653,018,119           NaN        355.7   
17        31          9   1,653,018,119           NaN        355.7   
18        48          9   1,653,018,119           NaN        355.7   
19        59          9   1,653,018,119           NaN        355.7   
20       311          9   1,653,018,119           NaN        355.7   
21       863          9   1,653,018,119           NaN        355.7   

   Release Date  Year  Month  Day                 Track  \
1    2024-05-04  2024      5    4           Not Like Us   
7    2024-05-31  2024      5   31               Houdini   
13   2024-02-02  2024      2    2             Gata O

In [80]:
# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up
respuesta_5 = cubo_olap.groupby('Year')['Track Score'].mean()
print(respuesta_5.head(10))

Year
1987    21.50
1991    19.60
1994    31.50
1998    20.80
1999    22.20
2000    31.70
2001    20.10
2002    25.85
2003    25.00
2004    23.60
Name: Track Score, dtype: float64


In [81]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
print(respuesta_6.head(10))

     id_track  id_tiempo Spotify Streams  YouTube Views  Track Score  \
155        64         57   1,811,990,630  1,516,863,120        184.8   
187        89         57   1,514,239,759  1,366,181,141        163.1   
201       103         91     478,150,858    113,379,734        150.7   
285       175         91     689,181,455    183,381,482        114.2   
300       188         57   1,421,763,497    945,857,087        110.7   
373       246        204     717,849,037    217,687,654         98.4   
374      2221        204     717,849,037    217,687,654         98.4   
391       262        135   1,977,947,460  1,861,597,184         96.1   
402       272         57   1,108,031,580    765,780,154         94.1   
423       291         57   1,293,773,697    501,908,214         90.1   

    Release Date  Year  Month  Day               Track     Artist  \
155   2022-05-06  2022      5    6     Me Porto Bonito  Bad Bunny   
187   2022-05-06  2022      5    6  Titï¿½ï¿½ Me Pregu  Bad Bunny   


In [82]:
# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-Down
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum()
print(respuesta_7.head(10))

Artist       Track                                                              
"XY"         Come and Say - Original Mix                                            415,159,148
$OHO BANI    ZEIT, DASS SICH WAS DREHT                                                6,744,201
$uicideboy$  ...And to Those I Love, Thanks for Sticking Around                     269,460,083
             1000 Blunts                                                             87,680,985
             Are You Going to See the Rose in the Vase, or the Dust on the Table      2,640,169
             Avalon                                                                  49,001,959
             For the Last Time                                                      149,247,747
             Matte Black                                                             82,186,979
             Not Even Ghosts Are This Empty                                          50,747,796
             The Thin Grey Line                        

In [83]:
# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice
respuesta_8 = cubo_olap[(cubo_olap['Month'] == 12 ) & (cubo_olap['Year'] == 2023)]
print(respuesta_8.head(10))

     id_track  id_tiempo Spotify Streams YouTube Views  Track Score  \
152        62         55           1,384       222,076        186.5   
193        95         84     649,404,552   433,839,212        155.9   
196        98         87     359,803,580   369,321,334        153.1   
235       132        112           8,327           NaN        130.7   
289       178        152     606,124,553   166,363,445        112.6   
290      1632        152     606,124,553   166,363,445        112.6   
347         8        186       4,674,619   108,313,521        103.1   
348       224        186       4,674,619   108,313,521        103.1   
372       245        203     298,853,393   112,961,806         98.5   
479       332        258      24,933,240    79,219,504         84.3   

    Release Date  Year  Month  Day                                     Track  \
152   2023-12-31  2023     12   31  Very Cute Melody by Marimba Tone (39813)   
193   2023-12-02  2023     12    2                        

In [84]:
# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up
respuesta_9 = cubo_olap.groupby('Month')['Spotify Streams'].sum()
print(respuesta_9)

Month
1     2,031,280,6332,031,280,6332,031,280,6332,031,2...
2     675,079,153675,079,153426,700,979426,700,97930...
3     601,309,283355,186,436355,186,436925,655,5693,...
4     390,470,936395,433,400547,882,871547,882,87154...
5     323,703,884107,034,922107,034,922107,034,92290...
6     1,653,018,1191,653,018,1191,653,018,1191,653,0...
7     3,107,100,3493,107,100,349670,376,179891,720,2...
8     1,065,399,3311,065,399,3311,065,399,3311,658,2...
9     1,258,569,6941,258,569,6941,258,569,694750,154...
10    297,1851,720,918,0651,563,224,246478,150,8583,...
11    670,665,438670,665,438493,627,071493,627,07149...
12    720,822,868720,822,868720,822,8681,337,619,917...
Name: Spotify Streams, dtype: object


In [85]:
# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice
respuesta_10 = cubo_olap[(cubo_olap['Artist'] == 'Taylor Swift' ) & (cubo_olap['Explicit Track'] == 0) & (cubo_olap['Year'] == 2024 ) ]
print(respuesta_10)

      id_track  id_tiempo Spotify Streams YouTube Views  Track Score  \
38          17         17     395,433,400   104,673,836        297.6   
397       2529         17     250,430,303    16,747,466         95.0   
676       3452         17     210,385,853    18,713,330         68.7   
907       2525        123      17,310,258    43,892,777         57.2   
1081       867         17     166,168,092    16,423,009         52.3   
1249      1014         17     175,613,480    15,237,337         47.8   
1431      1168         17     161,916,445    13,509,432         44.0   
1552      1271         17     165,611,453     9,873,283         42.1   
2956      2485        183      97,601,615     5,528,058         28.6   
2999      2518         17     106,201,248     5,521,347         28.3   
3014      2529         17     250,430,303           NaN         28.2   
3281      2777        183      88,244,010     7,381,186         26.7   
3325      2817         17     105,424,914     7,227,379         